<a href="https://colab.research.google.com/github/muasmuchtar/AGENDA-KERJA/blob/master/salin_folder_shared_gdrive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Salin Folder "Shared with Me" ke My Drive

Notebook ini menyalin folder yang di-share ke kamu (beserta seluruh isi sub-foldernya) menjadi milikmu sendiri di **My Drive**.

**Cara pakai:**
1. Jalankan Cell 1 (Autentikasi) — login dengan akun Google yang ingin dipakai.
2. Isi nama folder di Cell 2, lalu jalankan untuk mencarinya.
3. Jalankan Cell 3 untuk mendefinisikan fungsi penyalinan.
4. Jalankan Cell 4 untuk mulai menyalin.
5. Untuk akun Google lain, buka notebook ini lagi di jendela browser lain (atau incognito) yang login dengan akun berbeda, lalu ulangi langkah 1-4.

## 1. Autentikasi ke Google Drive

In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

from google.colab import auth
from googleapiclient.discovery import build
from google.auth import default

auth.authenticate_user()
creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)

print("\u2705 Berhasil login dan terhubung ke Google Drive!")

## 2. Cari folder di "Shared with me"

In [ ]:
folder_name = "Classroom"  #@param {type:"string"}

def find_shared_folder(name):
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        "and sharedWithMe = true and trashed = false"
    )
    results = drive_service.files().list(
        q=query,
        fields="files(id, name, owners)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()
    return results.get('files', [])

folders = find_shared_folder(folder_name)

if not folders:
    print(f"\u274c Folder '{folder_name}' tidak ditemukan di 'Shared with me'.")
    print("Pastikan nama folder persis sama (huruf besar/kecil dan spasi harus cocok).")
else:
    for f in folders:
        owner = f.get('owners', [{}])[0].get('emailAddress', 'tidak diketahui')
        print(f"\u2705 Ditemukan: {f['name']} (ID: {f['id']}) - pemilik: {owner}")
    source_folder_id = folders[0]['id']

## 3. Fungsi penyalinan (rekursif, termasuk sub-folder)

In [ ]:
def create_folder(name, parent_id=None):
    file_metadata = {
        'name': name,
        'mimeType': 'application/vnd.google-apps.folder'
    }
    if parent_id:
        file_metadata['parents'] = [parent_id]
    folder = drive_service.files().create(
        body=file_metadata, fields='id', supportsAllDrives=True
    ).execute()
    return folder.get('id')

def copy_file(file_id, new_name, parent_id):
    copied_metadata = {'name': new_name, 'parents': [parent_id]}
    return drive_service.files().copy(
        fileId=file_id, body=copied_metadata, supportsAllDrives=True
    ).execute()

def copy_folder_recursive(source_id, dest_parent_id, folder_name, indent=0):
    prefix = '  ' * indent
    print(f"{prefix}\U0001F4C1 Membuat folder: {folder_name}")
    new_folder_id = create_folder(folder_name, dest_parent_id)

    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{source_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
            pageToken=page_token
        ).execute()

        for item in response.get('files', []):
            if item['mimeType'] == 'application/vnd.google-apps.folder':
                copy_folder_recursive(item['id'], new_folder_id, item['name'], indent + 1)
            else:
                try:
                    print(f"{prefix}  \U0001F4C4 Menyalin: {item['name']}")
                    copy_file(item['id'], item['name'], new_folder_id)
                except Exception as e:
                    print(f"{prefix}  \u26A0\uFE0F Gagal menyalin {item['name']}: {e}")

        page_token = response.get('nextPageToken')
        if not page_token:
            break

    return new_folder_id

## 4. Jalankan penyalinan ke My Drive

In [ ]:
print(f"\U0001F680 Memulai penyalinan folder '{folder_name}' ke My Drive...\n")

new_folder_id = copy_folder_recursive(source_folder_id, None, folder_name)

print(f"\n\u2705 Selesai! Folder baru sudah ada di My Drive.")
print(f"\U0001F517 Link: https://drive.google.com/drive/folders/{new_folder_id}")

## Catatan
- Untuk akun Google lain, kamu perlu membuka notebook ini lagi (misalnya lewat jendela **incognito**) dan login dengan akun tersebut saat menjalankan Cell 1, lalu ulangi Cell 2–4.
- Google Docs, Sheets, dan Slides ikut tersalin dengan baik karena mendukung `files.copy` secara native.
- Untuk folder yang sangat besar (ribuan file), proses ini bisa memakan waktu karena setiap file disalin satu per satu mengikuti batas kuota API Google Drive.